# 🎼 Lesson 01: Fourier Transform — 傅里叶变换

**学习目标**
1. 理解什么是傅里叶变换
2. 学会使用 `fourier_pkg` 提供的工具
3. 通过可视化加深对频谱分析的理解

---

## 1. 什么是傅里叶变换？

> 傅里叶变换的核心思想：**任何周期信号都可以表示为一系列正弦波的叠加**

这就像把一首复杂的交响乐分解成各种乐器的声音——每种乐器产生特定频率的声音，傅里叶变换就是"听到"每个频率分量。

In [ ]:
# 导入我们创建的包
import numpy as np
import matplotlib.pyplot as plt

# 从 fourier_pkg 导入核心功能
from fourier_pkg import (
    generate_composite_signal,
    compute_fft,
    find_dominant_frequencies,
    nyquist_frequency,
    SignalGenerator,
    add_noise,
    estimate_snr,
    plot_combined_analysis,
    plot_signal_decomposition,
    plot_nyquist_demo,
)

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("✅ fourier_pkg 导入成功！")
print(f"   包版本: ", end="")
import fourier_pkg; print(fourier_pkg.__version__)

---

## 2. 创建一个复合信号

让我们创建一个由**三个不同频率**组成的复合信号：
- 5 Hz, 振幅 1.0
- 15 Hz, 振幅 0.5
- 50 Hz, 振幅 0.3

In [ ]:
# 参数设置
SAMPLE_RATE = 1000   # 采样率: 1000 Hz (每秒采样1000次)
DURATION = 1.0       # 信号持续时间: 1 秒

TRUE_FREQUENCIES = [5, 15, 50]    # 真实的频率成分
TRUE_AMPLITUDES = [1.0, 0.5, 0.3]
TRUE_PHASES = [0, np.pi/4, np.pi/2]

# 生成时间轴
t = np.arange(0, DURATION, 1/SAMPLE_RATE)

# 创建复合信号
signal = generate_composite_signal(t, TRUE_FREQUENCIES, TRUE_AMPLITUDES, TRUE_PHASES)

print(f"📊 信号参数:")
print(f"   采样率: {SAMPLE_RATE} Hz")
print(f"   时长: {DURATION} 秒")
print(f"   采样点数: {len(t)}")
print(f"   频率成分: {TRUE_FREQUENCIES} Hz")
print(f"   振幅: {TRUE_AMPLITUDES}")
print(f"   初相位: {TRUE_PHASES}")
print(f"\n   奈奎斯特频率: {nyquist_frequency(SAMPLE_RATE, len(t))} Hz")
print(f"   (理论上可分析的最高频率)")

---

## 3. 综合分析：四宫格可视化

In [ ]:
# 调用我们包里的综合分析函数
fig = plot_combined_analysis(
    t, signal, SAMPLE_RATE,
    true_frequencies=TRUE_FREQUENCIES,
    figsize=(14, 10)
)
plt.suptitle('傅里叶变换综合分析', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**🔍 解读分析结果：**

1. **时域图（右上）**：看到的是所有频率叠加后的波形，已经无法直接分辨出各个频率

2. **频谱图（左上）**：清晰显示了三个尖峰，正好对应我们设置的 5Hz、15Hz、50Hz

3. **相位谱（右下）**：显示各频率成分的相位关系

4. **频谱图（右上）**：显示频率随时间变化的"热图"

---

## 4. 信号分解：逐步展示每个频率成分

In [ ]:
# 使用包里的分解可视化函数
fig = plot_signal_decomposition(
    t, signal, 
    TRUE_FREQUENCIES, 
    TRUE_AMPLITUDES, 
    TRUE_PHASES,
    SAMPLE_RATE,
    figsize=(14, 12)
)
plt.suptitle('信号分解：展示每个频率成分', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**📌 关键观察：**

- 每个频率成分单独看都是纯净的正弦波
- 叠加后就变成了复杂的波形
- **频谱是理解信号本质的正确方式！**

---

## 5. 奈奎斯特采样定理演示

In [ ]:
# 演示欠采样（信号频率超过奈奎斯特频率）
# 60Hz 信号以 100Hz 采样 -> 发生混叠
fig = plot_nyquist_demo(sample_rate=100, signal_freq=60)
plt.suptitle('欠采样混叠现象演示', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n⚠️  警告：")
print(f"   信号频率 60Hz > 采样率/2 = 50Hz (Nyquist)")
print(f"   真实频率: 60 Hz")
print(f"   混叠后"看到的"频率: {abs(2 * (50 - 60))} Hz")
print(f"   这就是混叠/aliasing 现象！")

---

## 6. 噪声对频谱分析的影响

In [ ]:
# 添加高斯白噪声
noisy_signal = add_noise(signal, noise_level=0.1, seed=42)

# 计算SNR
snr = estimate_snr(signal, noisy_signal)
print(f"🔊 信噪比 (SNR): {snr:.2f} dB")

# 分析带噪声的信号
fig = plot_combined_analysis(
    t, noisy_signal, SAMPLE_RATE,
    true_frequencies=TRUE_FREQUENCIES,
    figsize=(14, 10)
)
plt.suptitle(f'带噪声信号的频谱分析 (SNR={snr:.1f} dB)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**🔬 观察：**
- 噪声在频谱中表现为均匀分布的"背景"
- 但3个主峰仍然清晰可辨
- 信噪比越低，峰值被噪声淹没越严重

---

## 7. 使用 SignalGenerator 工具类

In [ ]:
# SignalGenerator 是我们包里的便利工具
gen = SignalGenerator(sample_rate=1000, duration=1.0)

# 生成不同类型的测试信号
signals = {
    'Sine 5Hz': gen.sine(frequency=5, amplitude=1.0),
    'Square 2Hz': gen.square(frequency=2, amplitude=0.8),
    'Sawtooth 3Hz': gen.sawtooth(frequency=3, amplitude=0.6),
    'White Noise': gen.white_noise(amplitude=0.5, seed=42),
}

# 可视化不同信号
fig, axes = plt.subplots(4, 2, figsize=(14, 10))

for idx, (name, sig) in enumerate(signals.items()):
    # 时域
    axes[idx, 0].plot(gen.time, sig, linewidth=0.8)
    axes[idx, 0].set_title(f'{name} - 时域', fontweight='bold')
    axes[idx, 0].grid(True, alpha=0.3)
    
    # 频域
    freqs, mags = compute_fft(sig, gen.sample_rate)
    axes[idx, 1].stem(freqs, mags, linefmt='-', markerfmt=' ', basefmt='k-',
                      use_line_collection=True)
    axes[idx, 1].fill_between(freqs, mags, alpha=0.3)
    axes[idx, 1].set_title(f'{name} - 频域', fontweight='bold')
    axes[idx, 1].set_xlim([0, 100])
    axes[idx, 1].grid(True, alpha=0.3)

plt.suptitle('SignalGenerator: 多种信号类型', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 注意观察不同信号的频谱特征：")
print("   - 正弦波：单一频率尖峰")
print("   - 方波：基频 + 奇次谐波 (3f, 5f, 7f...)")
print("   - 锯齿波：基频 + 所有谐波")
print("   - 白噪声：所有频率均匀分布")

---

## 8. 关键函数回顾

In [ ]:
# 查看 fourier_pkg 包的公开 API
print("📦 fourier_pkg 公开 API:")
print("=" * 50)
for name in sorted(fourier_pkg.__all__):
    print(f"   • {name}")

print("\n" + "=" * 50)
print("\n🏗️  包结构:")
print("""
fourier_pkg/
├── __init__.py      # 公共 API 导出
├── core.py          # 核心 FFT 算法
├── utils.py         # 工具类和辅助函数
└── visualize.py     # 可视化函数
""")

---

## 🎯 练习题

1. **修改频率**：把 TRUE_FREQUENCIES 改成 [10, 30, 80]，重新运行分析

2. **添加噪声**：调节 `add_noise` 的 `noise_level` 参数，观察 SNR 变化对频谱的影响

3. **探索 SignalGenerator**：尝试生成 chirp（扫频）信号并分析其频谱

4. **思考题**：为什么方波的频谱只有奇次谐波？（提示：傅里叶级数）

---

**🎉 恭喜完成 Lesson 01！**